# SROIE Receipt CBA Experiment

**Domain**: Invoices/Receipts (SROIE ICDAR 2019)
**Method**: Vision extraction — LLM reads receipt images and maps values to canonical concepts
**Models**: Claude Haiku 4.5, GPT-4o-mini
**Scoring**: Value-first CBA matching

SROIE provides only 4 ground-truth concepts per receipt (company, address, date, total).
We expand to 14 extractable concepts across 3 families. CBA is measured against the 4 GT-backed concepts.
This tests whether models correctly bind the total amount to "total" vs "subtotal" vs "tax_amount", etc.

In [17]:
import subprocess, sys
for pkg in ["anthropic", "openai", "datasets", "Pillow"]:
    try:
        __import__(pkg if pkg != "Pillow" else "PIL")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--break-system-packages", "-q"])

import anthropic, openai, json, time, os, random, io, base64
from PIL import Image as PILImage
from datasets import load_dataset
from collections import defaultdict, Counter
from google.colab import userdata


ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print("Setup complete")

Setup complete


In [18]:
# Load SROIE from HuggingFace
ds = load_dataset("sizhkhy/SROIE")
train_data = ds["train"]
print(f"SROIE loaded: {len(train_data)} receipts")

# Parse entities from SROIE format
def parse_sroie_entities(rec):
    """Parse entities from sizhkhy/SROIE format."""
    fields = rec.get("fields", {})
    if isinstance(fields, str):
        fields = json.loads(fields)
    return {
        "company": fields.get("COMPANY", fields.get("company", "")),
        "date": fields.get("DATE", fields.get("date", "")),
        "address": fields.get("ADDRESS", fields.get("address", "")),
        "total": fields.get("TOTAL", fields.get("total", "")),
    }

# SROIE label -> canonical concept mapping
SROIE_TO_CANONICAL = {
    "company": "store_name",
    "address": "store_address",
    "date": "transaction_date",
    "total": "total",
}

# Show sample
sample = train_data[0]
entities = parse_sroie_entities(sample)
print(f"\nSample receipt 0:")
for k, v in entities.items():
    print(f"  {k}: {v}")

SROIE loaded: 626 receipts

Sample receipt 0:
  company: RELAIS TOTAL OULMES
  date: 30/03/2018
  address: AUTOROUTE RABAT MEKNES 15000 KHEMISSET
  total: 56.00


In [19]:
# Full extraction schema: 14 concepts, 3 families
RECEIPT_ONTOLOGY = {
    "families": {
        "merchant": ["store_name", "store_address", "store_city_state_zip", "store_phone"],
        "transaction": ["transaction_date", "transaction_time", "receipt_number", "cashier", "payment_method"],
        "amounts": ["subtotal", "tax_amount", "total", "amount_tendered", "change_due"],
    }
}

ALL_CONCEPTS = [c for fam in RECEIPT_ONTOLOGY["families"].values() for c in fam]
CONCEPT_TO_FAMILY = {c: f for f, cs in RECEIPT_ONTOLOGY["families"].items() for c in cs}

# GT-backed concepts (only these 4 have ground truth in SROIE)
GT_CONCEPTS = list(SROIE_TO_CANONICAL.values())

SYSTEM_PROMPT = """You are a receipt extraction system. Given a receipt image, extract values and map each to the correct canonical concept.

Canonical concepts:
- store_name: Name of the store or business
- store_address: Store street address
- store_city_state_zip: Store city, state/province, and postal code
- store_phone: Store phone number
- transaction_date: Date of the transaction
- transaction_time: Time of the transaction
- receipt_number: Receipt or transaction ID number
- cashier: Cashier name or ID
- payment_method: Payment method (cash, card, etc.)
- subtotal: Sum of items BEFORE tax
- tax_amount: Tax amount charged
- total: Final total amount INCLUDING tax
- amount_tendered: Amount the customer paid
- change_due: Change returned to customer

Return a JSON object with exactly these 14 keys.
Monetary values: numeric string without currency symbols or commas (e.g., "4.95").
Dates: preserve format as shown on receipt.
If a field is not present on the receipt, use "N/A".
Return ONLY valid JSON, no other text."""

print(f"Schema: {len(ALL_CONCEPTS)} concepts, {len(RECEIPT_ONTOLOGY['families'])} families")
print(f"GT-backed concepts: {GT_CONCEPTS}")

Schema: 14 concepts, 3 families
GT-backed concepts: ['store_name', 'store_address', 'transaction_date', 'total']


In [20]:
SAMPLE_SIZE = 50
random.seed(42)

# Save receipt images to /tmp for processing
IMG_DIR = "/tmp/sroie_images"
os.makedirs(IMG_DIR, exist_ok=True)

indices = list(range(len(train_data)))
random.shuffle(indices)
selected = sorted(indices[:SAMPLE_SIZE])

samples = []
for i, idx in enumerate(selected):
    rec = train_data[idx]
    entities = parse_sroie_entities(rec)

    # Build canonical ground truth (only 4 GT-backed concepts, rest N/A)
    gt = {c: "N/A" for c in ALL_CONCEPTS}
    for sroie_key, canonical_key in SROIE_TO_CANONICAL.items():
        val = entities.get(sroie_key, "")
        if val and str(val).strip():
            gt[canonical_key] = str(val).strip()

    # Save image
    doc_id = f"sroie_{i:04d}"
    img_path = os.path.join(IMG_DIR, f"{doc_id}.png")
    img = rec["images"]
    if isinstance(img, PILImage.Image):
        img.save(img_path)
    elif isinstance(img, bytes):
        with open(img_path, "wb") as f:
            f.write(img)
    elif isinstance(img, dict) and "bytes" in img:
        with open(img_path, "wb") as f:
            f.write(img["bytes"])

    samples.append({
        "doc_id": doc_id,
        "image_path": img_path,
        "source_index": idx,
        "ground_truth": gt,
        "sroie_entities": entities,
    })

# Count GT fill rates
filled = Counter()
for s in samples:
    for c in GT_CONCEPTS:
        if s["ground_truth"].get(c, "N/A") != "N/A":
            filled[c] += 1

print(f"Sampled {len(samples)} receipts")
print(f"\nGT fill rates:")
for c in GT_CONCEPTS:
    print(f"  {c}: {filled[c]}/{SAMPLE_SIZE}")

Sampled 50 receipts

GT fill rates:
  store_name: 50/50
  store_address: 50/50
  transaction_date: 50/50
  total: 50/50


In [21]:
def normalize_value(v):
    """Normalize a value for comparison."""
    v = str(v).strip().lower()
    v = v.replace("$", "").replace(",", "").replace("rm", "").replace("rp", "").strip()
    v = " ".join(v.split())  # normalize whitespace
    try:
        numeric = float(v)
        v = f"{numeric:.2f}"
    except ValueError:
        pass
    return v

def score_document(gt, pred):
    """Score only GT-backed concepts using value-first CBA matching."""
    pred_norm = {k: normalize_value(v) for k, v in pred.items()}
    pred_values = {v for v in pred_norm.values() if v != normalize_value("N/A")}

    # Build combined address from model predictions for address matching
    pred_addr = normalize_value(pred.get("store_address", "N/A"))
    pred_csz = normalize_value(pred.get("store_city_state_zip", "N/A"))
    pred_combined = ""
    parts = []
    if pred_addr != normalize_value("N/A"):
        parts.append(pred_addr)
    if pred_csz != normalize_value("N/A"):
        parts.append(pred_csz)
    if parts:
        pred_combined = " ".join(parts)

    per_field = {}
    field_correct = 0
    cba_correct = 0
    scored = []

    for concept in ALL_CONCEPTS:
        gt_val = normalize_value(gt.get(concept, "N/A"))
        pred_val = normalize_value(pred.get(concept, "N/A"))

        # Skip concepts without GT
        if gt_val == normalize_value("N/A"):
            continue

        scored.append(concept)

        # Special address handling: SROIE stores full address as one string
        if concept == "store_address":
            cba_match = (gt_val == pred_val) or (gt_val == pred_combined)
            field_match = (gt_val in pred_values) or (gt_val == pred_combined)
            # Also check containment
            if not field_match and pred_combined:
                field_match = (gt_val in pred_combined) or (pred_combined in gt_val)
            if not cba_match and pred_combined:
                cba_match = (gt_val in pred_combined) or (pred_combined in gt_val)
        else:
            cba_match = (gt_val == pred_val)
            field_match = (gt_val in pred_values)

        if cba_match:
            cba_correct += 1
        if field_match:
            field_correct += 1

        # Find where GT value was actually bound
        bound_to = None
        if field_match and not cba_match:
            for k, v in pred.items():
                if normalize_value(v) == gt_val and k != concept:
                    bound_to = k
                    break

        per_field[concept] = {
            "gt_value": gt_val,
            "pred_value": pred_val,
            "field_match": field_match,
            "cba_match": cba_match,
            "misbinding": field_match and not cba_match,
            "bound_to": bound_to,
        }

    total = len(scored)
    if total == 0:
        return {"field_recall": 0, "cba_strict": 0, "delta": 0, "total": 0, "per_field": {}, "misbinding_count": 0}

    misbinding_count = sum(1 for d in per_field.values() if d["misbinding"])

    return {
        "field_recall": field_correct / total,
        "cba_strict": cba_correct / total,
        "cba_soft": cba_correct / total,  # simplified for GT-only
        "delta": (field_correct - cba_correct) / total,
        "total": total,
        "scored_concepts": scored,
        "per_field": per_field,
        "misbinding_count": misbinding_count,
    }

# Sanity check
test_gt = {"store_name": "N/A", "store_address": "N/A", "transaction_date": "N/A", "total": "25.00",
           "subtotal": "N/A", "tax_amount": "N/A", "store_city_state_zip": "N/A", "store_phone": "N/A",
           "transaction_time": "N/A", "receipt_number": "N/A", "cashier": "N/A", "payment_method": "N/A",
           "amount_tendered": "N/A", "change_due": "N/A"}
test_pred_good = {c: "N/A" for c in ALL_CONCEPTS}
test_pred_good["total"] = "25.00"
test_pred_bad = {c: "N/A" for c in ALL_CONCEPTS}
test_pred_bad["subtotal"] = "25.00"  # misbinding: total value under subtotal

s_good = score_document(test_gt, test_pred_good)
s_bad = score_document(test_gt, test_pred_bad)
assert s_good["cba_strict"] == 1.0, f"Expected 1.0, got {s_good['cba_strict']}"
assert s_bad["misbinding_count"] == 1, f"Expected 1 misbinding, got {s_bad['misbinding_count']}"
print(f"Scoring functions verified (sanity checks passed)")

Scoring functions verified (sanity checks passed)


In [22]:
def encode_image(image_path, max_width=1024):
    """Load, resize, and base64-encode an image."""
    img = PILImage.open(image_path)
    if img.width > max_width:
        ratio = max_width / img.width
        img = img.resize((max_width, int(img.height * ratio)), PILImage.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.standard_b64encode(buf.getvalue()).decode("utf-8")

def extract_anthropic(image_path, model_id):
    """Extract via Anthropic vision API."""
    b64 = encode_image(image_path)
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    message = client.messages.create(
        model=model_id,
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": b64}},
                {"type": "text", "text": "Extract all fields from this receipt image."},
            ],
        }],
    )
    raw = message.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)

def extract_openai(image_path, model_id):
    """Extract via OpenAI vision API."""
    b64 = encode_image(image_path)
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model=model_id,
        max_tokens=1024,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": "Extract all fields from this receipt image."},
            ]},
        ],
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)

MODELS = {
    "haiku": {"provider": "anthropic", "model_id": "claude-haiku-4-5-20251001"},
    "gpt4o-mini": {"provider": "openai", "model_id": "gpt-4o-mini"},
}

def extract_receipt(image_path, model_name):
    """Route to correct provider."""
    cfg = MODELS[model_name]
    if cfg["provider"] == "anthropic":
        return extract_anthropic(image_path, cfg["model_id"])
    else:
        return extract_openai(image_path, cfg["model_id"])

print(f"Extraction functions ready. Models: {list(MODELS.keys())}")

Extraction functions ready. Models: ['haiku', 'gpt4o-mini']


In [29]:
all_results = {}

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Running: {model_name} ({MODELS[model_name]['model_id']})")
    print(f"{'='*60}")

    results = []
    errors = []

    for i, s in enumerate(samples):
        predicted = None
        for attempt in range(5):
            try:
                predicted = extract_receipt(s["image_path"], model_name)
                break
            except Exception as e:
                err_str = str(e)
                is_rate_limit = "429" in err_str or "rate_limit" in err_str.lower()
                if attempt < 4:
                    if is_rate_limit:
                        wait = min(10 * (2 ** attempt), 120)  # 10, 20, 40, 80s for rate limits
                        print(f"  [{s['doc_id']}] Rate limited (attempt {attempt+1}). Waiting {wait}s...")
                    else:
                        wait = 2 ** (attempt + 1)
                        print(f"  [{s['doc_id']}] Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"  [{s['doc_id']}] FAILED after 5 attempts: {e}")
                    errors.append({"doc_id": s["doc_id"], "error": str(e)})

        if predicted is None:
            continue

        scores = score_document(s["ground_truth"], predicted)
        results.append({
            "doc_id": s["doc_id"],
            "predicted": predicted,
            "ground_truth": s["ground_truth"],
            "scores": scores,
        })

        mb = scores["misbinding_count"]
        status = "OK" if mb == 0 else f"MISBIND={mb}"
        if (i + 1) % 10 == 0 or i == 0 or mb > 0:
            print(f"  [{i+1:2d}/{len(samples)}] {s['doc_id']} — F1={scores['field_recall']:.3f}  CBA={scores['cba_strict']:.3f}  {status}")

        # Longer sleep for OpenAI to avoid TPM rate limits on vision requests
        time.sleep(1.5 if MODELS[model_name]["provider"] == "anthropic" else 5.0)

    # Aggregate
    if results:
        avg_f1 = sum(r["scores"]["field_recall"] for r in results) / len(results)
        avg_cba = sum(r["scores"]["cba_strict"] for r in results) / len(results)
        avg_delta = sum(r["scores"]["delta"] for r in results) / len(results)
        total_misbindings = sum(r["scores"]["misbinding_count"] for r in results)
    else:
        avg_f1 = avg_cba = avg_delta = total_misbindings = 0

    all_results[model_name] = {
        "results": results,
        "errors": errors,
        "avg_f1": round(avg_f1, 4),
        "avg_cba": round(avg_cba, 4),
        "avg_delta": round(avg_delta, 4),
        "total_misbindings": total_misbindings,
    }

    print(f"\n--- {model_name} Summary ---")
    print(f"  Field Recall:     {avg_f1:.4f}")
    print(f"  CBA-strict:   {avg_cba:.4f}")
    print(f"  Delta:        {avg_delta:.4f}")
    print(f"  Misbindings:  {total_misbindings}")
    print(f"  Errors:       {len(errors)}")

print(f"\n{'='*60}")
print("ALL EXPERIMENTS COMPLETE")
print(f"{'='*60}")


Running: haiku (claude-haiku-4-5-20251001)
  [ 1/50] sroie_0000 — F1=0.750  CBA=0.750  OK
  [10/50] sroie_0009 — F1=0.750  CBA=0.750  OK
  [20/50] sroie_0019 — F1=0.750  CBA=0.750  OK
  [30/50] sroie_0029 — F1=1.000  CBA=1.000  OK
  [40/50] sroie_0039 — F1=0.750  CBA=0.750  OK
  [50/50] sroie_0049 — F1=0.500  CBA=0.500  OK

--- haiku Summary ---
  Field F1:     0.6850
  CBA-strict:   0.6850
  Delta:        0.0000
  Misbindings:  0
  Errors:       0

Running: gpt4o-mini (gpt-4o-mini)
  [ 1/50] sroie_0000 — F1=0.750  CBA=0.750  OK
  [ 2/50] sroie_0001 — F1=0.750  CBA=0.500  MISBIND=1
  [10/50] sroie_0009 — F1=0.750  CBA=0.750  OK
  [20/50] sroie_0019 — F1=0.750  CBA=0.750  OK
  [30/50] sroie_0029 — F1=0.750  CBA=0.750  OK
  [sroie_0036] Rate limited (attempt 1). Waiting 10s...
  [39/50] sroie_0038 — F1=1.000  CBA=0.750  MISBIND=1
  [40/50] sroie_0039 — F1=0.750  CBA=0.750  OK
  [sroie_0044] Rate limited (attempt 1). Waiting 10s...
  [50/50] sroie_0049 — F1=0.500  CBA=0.500  OK

--- gpt4

In [24]:
# Cross-model comparison
print("=" * 70)
print("CROSS-MODEL COMPARISON — SROIE Receipts (GT-only, 4 concepts)")
print("=" * 70)

print(f"\n{'Model':<15} {'Field Recall':>10} {'CBA-strict':>12} {'Delta':>8} {'Misbindings':>13} {'Errors':>8}")
print("-" * 70)
for name, res in all_results.items():
    print(f"{name:<15} {res['avg_f1']:>10.4f} {res['avg_cba']:>12.4f} {res['avg_delta']:>8.4f} {res['total_misbindings']:>13} {len(res['errors']):>8}")

total_misbindings = sum(r["total_misbindings"] for r in all_results.values())
print(f"\nTotal misbindings across all models: {total_misbindings}")

# Per-concept accuracy
print(f"\n{'='*70}")
print("PER-CONCEPT ACCURACY (across all models)")
print(f"{'='*70}")

concept_stats = defaultdict(lambda: {"cba": 0, "field": 0, "total": 0, "misbindings": 0})
for model_name, res in all_results.items():
    for r in res["results"]:
        for concept, detail in r["scores"]["per_field"].items():
            cs = concept_stats[concept]
            cs["total"] += 1
            if detail["cba_match"]:
                cs["cba"] += 1
            if detail["field_match"]:
                cs["field"] += 1
            if detail["misbinding"]:
                cs["misbindings"] += 1

print(f"\n{'Concept':<25} {'Family':<15} {'Field Acc':>10} {'CBA Acc':>10} {'Delta':>8} {'Misbindings':>13}")
print("-" * 85)
for concept in GT_CONCEPTS:
    cs = concept_stats[concept]
    if cs["total"] == 0:
        continue
    f_acc = cs["field"] / cs["total"]
    c_acc = cs["cba"] / cs["total"]
    delta = f_acc - c_acc
    fam = CONCEPT_TO_FAMILY[concept]
    print(f"{concept:<25} {fam:<15} {f_acc:>10.3f} {c_acc:>10.3f} {delta:>8.3f} {cs['misbindings']:>13}")

CROSS-MODEL COMPARISON — SROIE Receipts (GT-only, 4 concepts)

Model             Field F1   CBA-strict    Delta   Misbindings   Errors
----------------------------------------------------------------------
haiku               0.7050       0.7050   0.0000             0        0
gpt4o-mini          0.7200       0.7050   0.0150             3        0

Total misbindings across all models: 3

PER-CONCEPT ACCURACY (across all models)

Concept                   Family           Field Acc    CBA Acc    Delta   Misbindings
-------------------------------------------------------------------------------------
store_name                merchant             0.660      0.660    0.000             0
store_address             merchant             0.400      0.400    0.000             0
transaction_date          transaction          0.910      0.910    0.000             0
total                     amounts              0.880      0.850    0.030             3


In [25]:
# Misbinding confusion pairs
print("=" * 70)
print("MISBINDING CONFUSION PAIRS")
print("=" * 70)

confusion = Counter()
all_misbindings = []
for model_name, res in all_results.items():
    for r in res["results"]:
        for concept, detail in r["scores"]["per_field"].items():
            if detail["misbinding"] and detail["bound_to"]:
                confusion[(concept, detail["bound_to"])] += 1
                all_misbindings.append({
                    "model": model_name,
                    "doc_id": r["doc_id"],
                    "concept": concept,
                    "bound_to": detail["bound_to"],
                    "gt_value": detail["gt_value"],
                    "family_gt": CONCEPT_TO_FAMILY.get(concept, "?"),
                    "family_pred": CONCEPT_TO_FAMILY.get(detail["bound_to"], "?"),
                })

if confusion:
    print(f"\n{'Expected Concept':<25} {'Bound To':<25} {'Count':>6} {'Families':>20}")
    print("-" * 80)
    for (src, dst), count in confusion.most_common(20):
        fam_src = CONCEPT_TO_FAMILY.get(src, "?")
        fam_dst = CONCEPT_TO_FAMILY.get(dst, "?")
        same = "SAME" if fam_src == fam_dst else "CROSS"
        print(f"{src:<25} {dst:<25} {count:>6} {fam_src}->{fam_dst} ({same})")
else:
    print("No misbindings detected.")

# Per-model detail
print(f"\n{'='*70}")
print("PER-MODEL MISBINDING LOG")
print(f"{'='*70}")
for model_name, res in all_results.items():
    model_mbs = [mb for mb in all_misbindings if mb["model"] == model_name]
    print(f"\n--- {model_name} ({len(model_mbs)} misbindings) ---")
    for mb in model_mbs:
        print(f"  {mb['doc_id']}: GT[{mb['concept']}]=\"{mb['gt_value'][:40]}\" -> predicted as [{mb['bound_to']}]")

MISBINDING CONFUSION PAIRS

Expected Concept          Bound To                   Count             Families
--------------------------------------------------------------------------------
total                     subtotal                       2 amounts->amounts (SAME)
total                     amount_tendered                1 amounts->amounts (SAME)

PER-MODEL MISBINDING LOG

--- haiku (0 misbindings) ---

--- gpt4o-mini (3 misbindings) ---
  sroie_0001: GT[total]="8.20" -> predicted as [amount_tendered]
  sroie_0007: GT[total]="22.90" -> predicted as [subtotal]
  sroie_0038: GT[total]="3.90" -> predicted as [subtotal]


In [26]:
export = {
    "experiment": "hindsight_sroie_cba",
    "domain": "receipts",
    "dataset": "SROIE (ICDAR 2019)",
    "dataset_source": "sizhkhy/SROIE (HuggingFace)",
    "sample_size": len(samples),
    "total_concepts": len(ALL_CONCEPTS),
    "gt_backed_concepts": GT_CONCEPTS,
    "ontology": RECEIPT_ONTOLOGY,
    "models": {k: v for k, v in MODELS.items()},
    "scoring_method": "value-first CBA, GT-only (4 concepts)",
    "results": {
        model_name: {
            "field_recall": res["avg_f1"],
            "cba_strict": res["avg_cba"],
            "delta": res["avg_delta"],
            "total_misbindings": res["total_misbindings"],
            "num_errors": len(res["errors"]),
        }
        for model_name, res in all_results.items()
    },
    "confusion_pairs": [
        {"expected": src, "bound_to": dst, "count": count}
        for (src, dst), count in confusion.most_common(20)
    ] if confusion else [],
    "total_misbindings": sum(r["total_misbindings"] for r in all_results.values()),
}

output_path = "/tmp/hindsight_sroie_cba_results.json"
with open(output_path, "w") as f:
    json.dump(export, f, indent=2)
print(f"Results exported to: {output_path}")

# Cross-domain comparison
print(f"\n{'='*70}")
print("CROSS-DOMAIN COMPARISON (all experiments)")
print(f"{'='*70}")
print(f"\n{'Domain':<20} {'Dataset':<20} {'Misbindings':>12} {'Haiku Delta':>13} {'GPT4o-mini Delta':>17}")
print("-" * 85)
print(f"{'Paystubs':<20} {'Synthetic':<20} {'~5':>12} {'~0':>13} {'~0':>17}")
print(f"{'Receipts (CORD)':<20} {'CORD v2':<20} {'48':>12} {'-0.10':>13} {'~0':>17}")

haiku_d = all_results.get("haiku", {}).get("avg_delta", 0)
gpt_d = all_results.get("gpt4o-mini", {}).get("avg_delta", 0)
total_mb = sum(r["total_misbindings"] for r in all_results.values())
print(f"{'Receipts (SROIE)':<20} {'SROIE':<20} {total_mb:>12} {haiku_d:>13.3f} {gpt_d:>17.3f}")

print(f"{'Employment Law':<20} {'LEDGAR':<20} {'350':>12} {'+0.207':>13} {'+0.260':>17}")
print(f"{'Commercial Law':<20} {'CUAD v1':<20} {'434':>12} {'+0.244':>13} {'+0.335':>17}")

Results exported to: /tmp/hindsight_sroie_cba_results.json

CROSS-DOMAIN COMPARISON (all experiments)

Domain               Dataset               Misbindings   Haiku Delta  GPT4o-mini Delta
-------------------------------------------------------------------------------------
Paystubs             Synthetic                      ~5            ~0                ~0
Receipts (CORD)      CORD v2                        48         -0.10                ~0
Receipts (SROIE)     SROIE                           3         0.000             0.015
Employment Law       LEDGAR                        350        +0.207            +0.260
Commercial Law       CUAD v1                       434        +0.244            +0.335
